# Main

In [147]:
# importing packages
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [157]:

# -------------------------------------------------
# Define absolute paths to the three source files
# -------------------------------------------------
ROOT = "/Users/marierobillon/LocalFiles/DS_causal_inf_for_sustainability_teamwork"
RAW = f"{ROOT}/data/data/raw"
ELCOM_CSV   = f"{RAW}/elcom_tariffs_H2_H4_H7.csv"
BFE_PLANTS  = f"{RAW}/bfe_plants.csv/ElectricityProductionPlant.csv"
DWELLINGS   = f"{RAW}/bfs_dwellings_by_occupant_2000_all.csv"
CROSSWALK  = f"{ROOT}/data/data/processed/bfs_commune_mutations.csv"

# Data

## Load and first cleaning

### X variable

In [149]:

# Load **X** – ElCom household tariffs
# -------------------------------------------------
elcom_raw = pd.read_csv(ELCOM_CSV, sep=",", low_memory=False)

elcom_raw.head()


,period,municipality,operator,category,total,energy,grid,charge,aidfee,fixcosts
0,2018,3921,10,H2,25.359,7.765,13.194,2.10,2.3,108.0
1,2018,4045,804,H2,17.801,4.264,10.387,0.85,2.3,102.0
2,2018,5590,698,H2,25.480,8.900,12.350,1.93,2.3,60.0
3,2018,979,559,H2,22.802,6.763,12.989,0.75,2.3,126.0
4,2018,4080,594,H2,20.678,4.798,12.230,1.35,2.3,147.6


In [150]:
# Keep only the columns we need for X
# -----------------------------------
cols_needed = ["municipality", "period", "operator", "category", "total", "energy"]
elcom = elcom_raw[cols_needed].copy()

# Rename to the semantic names we’ll use later
elcom = elcom.rename(columns={
    "municipality":   "commune_number",   # old commune ID
    "period":         "year",             # tariff year (int)
    "operator":       "grid_operator",    # operator code
    "category":       "tariff_profile",   # H2 / H4 / H7, H4 correspond to swiss households
    "total":          "total_ct_per_kwh", # total H4 tariff (cent/kWh), sum of energy, grid, charge and aidfree colums
    "energy":         "energy_ct_per_kwh" # energy component (cent/kWh)
})

elcom.head()

,commune_number,year,grid_operator,tariff_profile,total_ct_per_kwh,energy_ct_per_kwh
0,3921,2018,10,H2,25.359,7.765
1,4045,2018,804,H2,17.801,4.264
2,5590,2018,698,H2,25.480,8.900
3,979,2018,559,H2,22.802,6.763
4,4080,2018,594,H2,20.678,4.798


In [151]:
# Only keeping the tariff_profile H4 because it corresponds to swiss households
# ----------------------------------------------------------------------------

elcom_h4 = elcom[elcom["tariff_profile"] == "H4"].copy()

print(elcom.shape)
print(elcom_h4.shape)


(126086, 6)
(42044, 6)


### Y Variable

In [152]:
# -------------------------------------------------
# Load **Y** – SFOE plant register (solar plants only)
# -------------------------------------------------
plants = pd.read_csv(BFE_PLANTS, low_memory=False)

plants.head()


,xtf_id,Address,PostCode,Municipality,Canton,EGID,BeginningOfOperation,InitialPower,TotalPower,MainCategory,SubCategory,PlantCategory,_x,_y
0,5686,Binenweg 5,3904,Naters,VS,892091.0,1969-09-01,349576.0,349576.0,maincat_1,subcat_1,plantcat_7,2642480.025,1130235.125
1,5726,Robbia 2,7741,S. Carlo (Poschiavo),GR,9080323.0,2024-01-16,38250.0,38250.0,maincat_1,subcat_1,plantcat_4,NaN,NaN
2,5727,NaN,7744,Campocologno,GR,NaN,1907-03-01,55000.0,55000.0,maincat_1,subcat_1,plantcat_7,2808646.000,1123676.000
3,5730,Büdemji,7240,Küblis,GR,NaN,1922-01-01,44200.0,44200.0,maincat_1,subcat_1,plantcat_7,2778481.000,1198505.000
4,5740,Auenstrasse,8783,Linthal,GL,NaN,1962-03-18,1526000.0,1526000.0,maincat_1,subcat_1,plantcat_6,NaN,NaN


In [153]:
# Keep only the columns we need for Y
# -----------------------------------

renaming_map = {
    "xtf_id"               : "plant_id",          # unique identifier for the plant
    "Address"              : "address",           # street address of the plant
    "PostCode"             : "postcode",          # postal code
    "Municipality"         : "commune_number",    # old commune ID (matches tariff & census)
    "Canton"               : "canton",            # Canton abbreviation (e.g. VS, GR)
    "EGID"                 : "egid",              # official land‑registry identifier
    "BeginningOfOperation": "commission_date",# date the plant entered service
    "InitialPower"         : "initial_power",     # power at start (kW) – rarely used
    "TotalPower"           : "capacity_kw",       # current installed capacity (kW)
    "MainCategory"         : "main_category",     # high‑level plant class
    "SubCategory"          : "sub_category",      # detailed plant class (subcat_2 = solar)
    "PlantCategory"        : "plant_category",    # mounting type (attached, integrated, free‑standing)
    "_x"                   : "coord_x_lv95",      # Swiss LV95 easting
    "_y"                   : "coord_y_lv95"       # Swiss LV95 northing
}
plants = plants.rename(columns=renaming_map)


In [154]:
# Extract year from commissioning date (format: YYYY‑MM‑DD or YYYY)
plants["commission_year"] = pd.to_datetime(plants["commission_date"], errors="coerce").dt.year
# We need: commune_number (or postcode → commune), commissioning year,capacity (kW), and coordinates (_x, _y) for spatial checks.
cols_needed_y = ["commune_number", "commission_year", "capacity_kw", "coord_x_lv95", "coord_y_lv95","sub_category"]
plants = plants[cols_needed_y]

plants.head()

,commune_number,commission_year,capacity_kw,coord_x_lv95,coord_y_lv95,sub_category
0,Naters,1969,349576.0,2642480.025,1130235.125,subcat_1
1,S. Carlo (Poschiavo),2024,38250.0,NaN,NaN,subcat_1
2,Campocologno,1907,55000.0,2808646.000,1123676.000,subcat_1
3,Küblis,1922,44200.0,2778481.000,1198505.000,subcat_1
4,Linthal,1962,1526000.0,NaN,NaN,subcat_1


In [181]:
solar["commune_number"]

62        Plan-les-Ouates
63         Stein am Rhein
77                Lostorf
78                Lostorf
79            Stüsslingen
               ...       
339494           Pampigny
339495       Colombier NE
339496         Geroldswil
339497             Blonay
339498         Nürensdorf
Name: commune_number, Length: 337455, dtype: str

In [83]:
# Keep only solar plants (SubCategory code = 'subcat_2')
solar = plants[plants["sub_category"] == "subcat_2"].copy()

print(plants.shape)
print(solar.shape)


(339499, 6)
(337455, 6)


### Z variable

For each commune, the number of dwellings for a specific number of rooms and occupancy

In [112]:
# -------------------------------------------------
#  Load **Z** – Census 2000 dwellings by occupant
# -------------------------------------------------
dwell = pd.read_csv(DWELLINGS, low_memory=False)

dwell.head()


,Gemeinde,Anzahl Zimmer,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in
0,Schweiz,1 Zimmer,139953,4015,5579,3752,1366,757,916,190
1,Schweiz,2 Zimmer,327795,15544,18692,13618,5510,1149,4561,466
2,0001 Aeugst am Albis,1 Zimmer,15,0,1,1,1,1,0,0
3,0001 Aeugst am Albis,2 Zimmer,53,0,1,1,0,0,0,0
4,0002 Affoltern am Albis,1 Zimmer,81,6,3,5,0,0,0,0


In [113]:
# Romoving the lines with the sum of specific dwellings for whole switzerland
dwell = dwell[dwell["Gemeinde"] != "Schweiz"]

# Extract only the 4‑digit code from the combined string
dwell["commune_number"] = dwell["Gemeinde"].str.extract(r"(\d{4})").astype(int)
dwell = dwell.drop(columns=["Gemeinde"])

dwell.head()


,Anzahl Zimmer,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,commune_number
2,1 Zimmer,15,0,1,1,1,1,0,0,1
3,2 Zimmer,53,0,1,1,0,0,0,0,1
4,1 Zimmer,81,6,3,5,0,0,0,0,2
5,2 Zimmer,396,36,10,8,0,2,2,0,2
6,1 Zimmer,16,0,2,3,1,0,0,0,3


In [114]:
# Compute the total dwellings per commune
# ---------------------------------------
# We keep the nine count columns (exclude the first and the last):
count_cols = [
    "Mieter/in",
    "Genossenschafter/in",
    "Stockwerk-/Wohnungseigentümer/in",
    "Alleineigentümer/in des Hauses",
    "Miteigentümer/in des Hauses",
    "Inhaber/in einer Dienstwohnung",
    "Inhaber/in einer Freiwohnung",
    "Pächter/in"
]

# sum all rows that belong to the same commune
# This collapses the “by‑room‑size” dimension.
dwell_agg = (
    dwell.groupby("commune_number", as_index=False)[count_cols]
          .sum()
)

# Row‑wise total dwellings for each commune (after aggregation)
dwell_agg["total_dwellings_2000"] = dwell_agg[count_cols].sum(axis=1)

dwell_agg.head()

,commune_number,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,total_dwellings_2000
0,1,225,0,55,266,29,5,2,6,588
1,2,2596,247,225,804,143,25,12,4,4056
2,3,701,64,140,495,59,11,10,1,1481
3,4,494,2,85,475,64,3,10,10,1143
4,5,499,51,81,444,56,4,5,3,1143


In [115]:
#  Compute Owner‑occupied count and share (Z variable)
# ------------------------------------------------------------------
dwell_agg["owner_occupied_count"] = (
    dwell_agg["Alleineigentümer/in des Hauses"] + dwell_agg["Miteigentümer/in des Hauses"]
)
dwell_agg["owner_occupied_share"] = (
    dwell_agg["owner_occupied_count"] / dwell_agg["total_dwellings_2000"]
)

dwell_agg.head()

,commune_number,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,total_dwellings_2000,owner_occupied_count,owner_occupied_share
0,1,225,0,55,266,29,5,2,6,588,295,0.501701
1,2,2596,247,225,804,143,25,12,4,4056,947,0.233481
2,3,701,64,140,495,59,11,10,1,1481,554,0.374072
3,4,494,2,85,475,64,3,10,10,1143,539,0.471566
4,5,499,51,81,444,56,4,5,3,1143,500,0.437445


In [117]:
# Only keep the columns of Z needed
# ---------------------------------

cols_needed_z = ["total_dwellings_2000", "owner_occupied_count", "owner_occupied_share", "commune_number"]
dwell_agg = dwell_agg[cols_needed_z]

dwell_agg.head()

,total_dwellings_2000,owner_occupied_count,owner_occupied_share,commune_number
0,588,295,0.501701,1
1,4056,947,0.233481,2
2,1481,554,0.374072,3
3,1143,539,0.471566,4
4,1143,500,0.437445,5


## Prepare to join datasets by updated commune numbers

In [158]:
# -------------------------------------------------
# Helper: map old commune numbers → 2026 numbers
# -------------------------------------------------

cross = pd.read_csv(CROSSWALK, low_memory=False) # contains the key for commune conversion from historical to new

cross.columns = (
    cross.columns.str.strip()
                 .str.lower()
                .str.replace(" ", "_")
                .str.replace(r"[^\w]", "", regex=True)
)

cross.head()


,mutationnumber,mutationdate,initialhistoricalcode,initialcode,initialname,initialparenthistoricalcode,initialparentname,initialstep,terminalhistoricalcode,terminalcode,terminalname,terminalparenthistoricalcode,terminalparentname,terminalstep
0,2773,01.01.2010,10948,343,Untersteckholz,10072,Amtsbezirk Aarwangen,29,14975,329,Langenthal,10286,Verwaltungskreis Oberaargau,26
1,2773,01.01.2010,12627,329,Langenthal,10072,Amtsbezirk Aarwangen,26,14975,329,Langenthal,10286,Verwaltungskreis Oberaargau,26
2,2774,01.01.2010,13366,531,Ballmoos,10216,Amtsbezirk Fraubrunnen,29,14976,540,Jegenstorf,10288,Verwaltungskreis Bern-Mittelland,26
3,2774,01.01.2010,13375,540,Jegenstorf,10216,Amtsbezirk Fraubrunnen,26,14976,540,Jegenstorf,10288,Verwaltungskreis Bern-Mittelland,26
4,2775,01.01.2010,13423,601,Aeschlen,10213,Amtsbezirk Konolfingen,29,14977,619,Oberdiessbach,10288,Verwaltungskreis Bern-Mittelland,26


In [161]:
# Creating a dictionary to change commune code
# --------------------------------------------

# Keep only rows where a real change happened
changed = cross[cross["initialcode"] != cross["terminalcode"]].copy()

# Create a mapping: old number → newest number
# A commune can appear more than once (multiple mutations over the years).
# We want the *latest* TerminalCode for each InitialCode.
latest_map = (
    changed.sort_values(["mutationnumber"], ascending=True)      # keep order of events
           .groupby("initialcode")["terminalcode"]
           .last()                                            # last mutation wins
           .to_dict()
)
# Example output (first few entries)
print({k: latest_map[k] for k in list(latest_map)[:5]})
# {2773: 329, 2774: 540, 2775: 601, …}

{21: 291, 30: 291, 32: 291, 36: 292, 42: 292}


In [175]:
# ------------------------------------------------------------------
# Trying the dictionary with the elcom dataset
# ------------------------------------------------------------------
elcom_h4["commune_no_new"] = (
    elcom_h4["commune_number"]
    .map(latest_map)
    .fillna(elcom_h4["commune_number"])
    .astype(int)
)
elcom_h4.head()

,commune_number,year,grid_operator,tariff_profile,total_ct_per_kwh,energy_ct_per_kwh,commune_no_new
7981,4726,2018,131,H4,23.305,6.313,4726
7982,177,2018,591,H4,15.649,4.568,177
7983,762,2018,274,H4,21.098,7.092,762
7984,4009,2018,582,H4,16.790,5.950,4009
7985,3701,2018,531,H4,12.009,5.709,3701


In [176]:
elcom_h4["commune_number"].equals(elcom_h4["commune_no_new"])

False

In [ ]:
# Taking a look at which values have changed

elcom_h4.loc[
    elcom_h4["commune_number"] != elcom_h4["commune_no_new"],
    ["commune_number", "commune_no_new"]
].value_counts().head(20)

commune_number  commune_no_new
3378            3396              49
6412            6458              22
3932            3901              20
6431            6436              19
6459            6513              18
700             6831              16
1057            1065              16
4179            4186              16
6454            6513              16
4122            4095              16
6744            6743              16
2456            2465              16
133             295               16
5078            5079              16
5064            5073              16
5181            5395              16
2016            2056              16
2278            2262              16
2027            2056              16
5146            5395              16
Name: count, dtype: int64

In [179]:
# ------------------------------------------------------------------
# Trying the dictionary with the dwell_agg dataset
# ------------------------------------------------------------------

dwell_agg["commune_no_new"] = (
    dwell_agg["commune_number"]
    .map(latest_map)
    .fillna(dwell_agg["commune_number"])
    .astype(int)
)
dwell_agg.head()

,total_dwellings_2000,owner_occupied_count,owner_occupied_share,commune_number,commune_no_new
0,588,295,0.501701,1,1
1,4056,947,0.233481,2,2
2,1481,554,0.374072,3,3
3,1143,539,0.471566,4,4
4,1143,500,0.437445,5,5


In [180]:
dwell_agg["commune_number"].equals(dwell_agg["commune_no_new"])

False

In [182]:
# Taking care of the Y variables which has communes names instead of number for the commune_number column
# -------------------------------------------------------------------------------------------------------

name_map = {}

# historical names
name_map.update(
    cross[["initialname", "terminalcode"]]
    .drop_duplicates()
    .set_index("initialname")["terminalcode"]
    .to_dict()
)

# current names
name_map.update(
    cross[["terminalname", "terminalcode"]]
    .drop_duplicates()
    .set_index("terminalname")["terminalcode"]
    .to_dict()
)

solar["commune_no_new"] = solar["commune_number"].map(name_map)

solar.head()

,commune_number,commission_year,capacity_kw,coord_x_lv95,coord_y_lv95,sub_category,commune_no_new
62,Plan-les-Ouates,2007,572.3,2496890.919,1113467.012,subcat_2,NaN
63,Stein am Rhein,2008,4.0,2706196.355,1280350.616,subcat_2,NaN
77,Lostorf,2003,6.4,2638166.304,1247780.316,subcat_2,NaN
78,Lostorf,1998,3.0,2638287.063,1247899.707,subcat_2,NaN
79,Stüsslingen,1999,7.7,2639924.966,1249094.349,subcat_2,2499.0


In [183]:
solar["commune_no_new"].isna().sum()

np.int64(244299)

In [186]:
solar_names = set(solar["commune_number"].dropna().str.strip())
cross_names = set(cross["terminalname"].dropna().str.strip())

print("Solar unique names:", len(solar_names))
print("Cross unique names:", len(cross_names))
print("Common names:", len(solar_names & cross_names))

Solar unique names: 3894
Cross unique names: 851
Common names: 621


In [187]:
cross[["terminalcode", "terminalname"]].drop_duplicates().shape

(852, 2)

## Compute Variables

### X variable

In [ ]:
# Compute **X** – Change in H4 total tariff (2023 – 2022)
# -------------------------------------------------

# First average tariff across operators for each commune‑year.
x_avg = (
    elcom_h4.groupby(["commune_number", "year"])["total_ct_per_kwh"]
         .mean()
         .reset_index(name="total_avg")
)

# Pivot to have 2022 and 2023 side‑by‑side
x_pivot = x_avg.pivot(index="commune_number", columns="year", values="total_avg")

# Keep only 2022 & 2023 (if missing, result will be NaN)
x_pivot = x_pivot[[2022, 2023]] if {2022, 2023}.issubset(x_pivot.columns) else x_pivot

# X = 2023 – 2022
x_pivot["X"] = x_pivot[2023] - x_pivot[2022]
X = x_pivot["X"].rename("X")          # Series indexed by commune_id

x_pivot.head()

### Y Variable

In [ ]:

# -------------------------------------------------
# 8️⃣  Compute **Y** – New solar installations per commune‑year
# -------------------------------------------------
y_counts = (
    solar.groupby(["commune_id", "commission_year"])
         .size()
         .reset_index(name="new_solar_plants")
)
# If you also want total capacity per year:
y_capacity = (
    solar.groupby(["commune_id", "commission_year"])["capacity_kw"]
         .sum()
         .reset_index(name="solar_capacity_kw")
)
Y = y_counts.pivot(index="commune_id", columns="commission_year", values="new_solar_plants")
# Optional: keep capacity side‑by‑side as a second DataFrame
Y_capacity = y_capacity.pivot(index="commune_id", columns="commission_year", values="solar_capacity_kw")

# -------------------------------------------------
# 10️⃣  Assemble a tidy DataFrame with X, Y, Z side‑by‑side
# -------------------------------------------------
final = pd.concat([X, Z], axis=1)               # X and Z are Series
# For Y we keep the yearly counts; you can also collapse to a single metric.
# Example: total new solar plants from 2020‑2026
if Y is not None:
    y_total = Y.loc[:, 2020:2026].sum(axis=1).rename("Y_total_new_solar_2020_2026")
    final = final.join(y_total, how="left")
print("✅ Finished loading X, Y, Z")
display(final.head())